In [1]:
!pip install gcsfs pyarrow pandas h3 --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# Cell 2 — Load January 2023 (1 bulan saja untuk prototype)
import pandas as pd
import gcsfs

fs = gcsfs.GCSFileSystem()

GCS_PATH = "hardy-geo-de-267342/raw/tlc_yellow/year=2023/month=01/yellow_tripdata_2023-01.parquet"

df = pd.read_parquet(f"gs://{GCS_PATH}", filesystem=fs)

print(f"Row count: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Row count: 3,066,766
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', 'year', 'month']
Memory usage: 453.6 MB


In [3]:
# Cell 3 — Apply trip filters (D-040 Standard Clean)
row_before = len(df)

df_clean = df[
    # Null checks datetime
    df["tpep_pickup_datetime"].notna() &
    df["tpep_dropoff_datetime"].notna() &
    # Null checks location
    df["PULocationID"].notna() &
    df["DOLocationID"].notna() &
    # No time travel + max 3 hours
    (df["tpep_dropoff_datetime"] > df["tpep_pickup_datetime"]) &
    ((df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() <= 10800) &
    # Valid zones (exclude Unknown 264, 265)
    (~df["PULocationID"].isin([264, 265])) &
    (~df["DOLocationID"].isin([264, 265])) &
    # Passenger count (cast to int first)
    (df["passenger_count"].astype("Int64") >= 1) &
    (df["passenger_count"].astype("Int64") <= 8) &
    # Distance
    (df["trip_distance"] >= 0.1) &
    (df["trip_distance"] <= 100) &
    # Fare
    (df["fare_amount"] >= 2.50) &
    (df["fare_amount"] <= 500) &
    # Total amount
    (df["total_amount"] >= 2.50) &
    (df["total_amount"] <= 500)
]

row_after = len(df_clean)
dropped = row_before - row_after
pct_dropped = (dropped / row_before) * 100

print(f"Before : {row_before:,}")
print(f"After  : {row_after:,}")
print(f"Dropped: {dropped:,} ({pct_dropped:.1f}%)")

Before : 3,066,766
After  : 2,823,775
Dropped: 242,991 (7.9%)


In [6]:
# Cell 4 — Download via gcsfs, save local, load dengan GeoPandas
import geopandas as gpd
import json

# Baca via gcsfs (sudah authenticated)
with fs.open("hardy-geo-de-267342/raw/reference/nyc_taxi_zones.geojson", "r") as f:
    geojson_content = f.read()

# Save ke local
with open("/tmp/nyc_taxi_zones.geojson", "w") as f:
    f.write(geojson_content)

# Load dari local
gdf_zones = gpd.read_file("/tmp/nyc_taxi_zones.geojson")

print(f"Zones loaded: {len(gdf_zones)} polygons")
print(f"CRS: {gdf_zones.crs}")
print(gdf_zones[["locationid", "zone", "borough"]].head())

Zones loaded: 263 polygons
CRS: EPSG:4326
  locationid                     zone        borough
0          1           Newark Airport            EWR
1          2              Jamaica Bay         Queens
2          3  Allerton/Pelham Gardens          Bronx
3          4            Alphabet City      Manhattan
4          5            Arden Heights  Staten Island


In [7]:
# Cell 5 — Cek columns yang ada
print(gdf_zones.columns.tolist())
print(gdf_zones.head(2))

['shape_area', 'locationid', 'shape_leng', 'zone', 'borough', 'geometry']
         shape_area locationid      shape_leng            zone borough  \
0   0.0007823067885          1  0.116357453189  Newark Airport     EWR   
1  0.00486634037837          2   0.43346966679     Jamaica Bay  Queens   

                                            geometry  
0  MULTIPOLYGON (((-74.18445 40.695, -74.18449 40...  
1  MULTIPOLYGON (((-73.82338 40.63899, -73.82277 ...  


In [8]:
# Cell 6 — Build LocationID → H3 lookup table
import h3

def get_h3_for_zone(geometry, resolution=8):
    """Get representative H3 cell untuk sebuah zone polygon."""
    # Gunakan centroid sebagai representative point
    centroid = geometry.centroid
    return h3.latlng_to_cell(centroid.y, centroid.x, resolution)

# Build lookup table
lookup_df = gdf_zones[["locationid", "zone", "borough"]].copy()
lookup_df["h3_res8"] = gdf_zones["geometry"].apply(get_h3_for_zone)
lookup_df = lookup_df.rename(columns={"locationid": "LocationID"})

print(f"Lookup table rows: {len(lookup_df)}")
print(f"Unique H3 cells: {lookup_df['h3_res8'].nunique()}")
print(lookup_df.head(5))

Lookup table rows: 263
Unique H3 cells: 247
  LocationID                     zone        borough          h3_res8
0          1           Newark Airport            EWR  882a1071e7fffff
1          2              Jamaica Bay         Queens  882a103955fffff
2          3  Allerton/Pelham Gardens          Bronx  882a10011dfffff
3          4            Alphabet City      Manhattan  882a100d31fffff
4          5            Arden Heights  Staten Island  882a1060e1fffff


In [9]:
# Cell 7 — Fix: cast LocationID ke int dulu
lookup_df["LocationID"] = lookup_df["LocationID"].astype("int64")

# Broadcast join untuk pickup
df_enriched = df_clean.merge(
    lookup_df[["LocationID", "h3_res8"]].rename(columns={"h3_res8": "h3_pickup_res8"}),
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
).drop(columns=["LocationID"])

# Broadcast join untuk dropoff
df_enriched = df_enriched.merge(
    lookup_df[["LocationID", "h3_res8"]].rename(columns={"h3_res8": "h3_dropoff_res8"}),
    left_on="DOLocationID",
    right_on="LocationID",
    how="left"
).drop(columns=["LocationID"])

# Derive day_of_week dan hour_of_day
df_enriched["day_of_week"] = df_enriched["tpep_pickup_datetime"].dt.dayofweek  # 0=Monday
df_enriched["hour_of_day"] = df_enriched["tpep_pickup_datetime"].dt.hour

# Drop Hive partition columns
df_enriched = df_enriched.drop(columns=["year", "month"])

print(f"Enriched rows: {len(df_enriched):,}")
print(f"Columns: {list(df_enriched.columns)}")
print(f"Null h3_pickup : {df_enriched['h3_pickup_res8'].isna().sum():,}")
print(f"Null h3_dropoff: {df_enriched['h3_dropoff_res8'].isna().sum():,}")

Enriched rows: 2,824,855
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', 'h3_pickup_res8', 'h3_dropoff_res8', 'day_of_week', 'hour_of_day']
Null h3_pickup : 4
Null h3_dropoff: 32


In [10]:
# Cell 8 — Investigasi duplicate dan nulls

# 1. Cek duplicate di lookup table
print("=== Duplicate LocationID di lookup ===")
print(lookup_df[lookup_df["LocationID"].duplicated(keep=False)].sort_values("LocationID"))

# 2. Cek LocationID yang produce null H3
print("\n=== LocationID yang null h3_pickup ===")
null_pu = df_enriched[df_enriched["h3_pickup_res8"].isna()]["PULocationID"].unique()
print(f"LocationIDs: {null_pu}")

print("\n=== LocationID yang null h3_dropoff ===")
null_do = df_enriched[df_enriched["h3_dropoff_res8"].isna()]["DOLocationID"].unique()
print(f"LocationIDs: {null_do}")

=== Duplicate LocationID di lookup ===
     LocationID                                           zone    borough  \
55           56                                         Corona     Queens   
56           56                                         Corona     Queens   
102         103  Governor's Island/Ellis Island/Liberty Island  Manhattan   
103         103  Governor's Island/Ellis Island/Liberty Island  Manhattan   
104         103  Governor's Island/Ellis Island/Liberty Island  Manhattan   

             h3_res8  
55   882a100c4bfffff  
56   882a100c49fffff  
102  882a1072b5fffff  
103  882a1072a3fffff  
104  882a1072b9fffff  

=== LocationID yang null h3_pickup ===
LocationIDs: [57]

=== LocationID yang null h3_dropoff ===
LocationIDs: [ 57 105]


In [11]:
# Cell 9 — Fix duplicate dan null

# 1. Deduplicate lookup: ambil satu H3 per LocationID (first occurrence)
lookup_deduped = lookup_df.drop_duplicates(subset=["LocationID"], keep="first")
print(f"Lookup before dedup: {len(lookup_df)}")
print(f"Lookup after dedup : {len(lookup_deduped)}")

# 2. Cek LocationID 57 dan 105 — ada di mana?
print("\n=== LocationID 57 dan 105 di GeoJSON ===")
print(gdf_zones[gdf_zones["locationid"].astype(int).isin([57, 105])][["locationid", "zone", "borough"]])

# 3. Cek di trip data — berapa banyak trips affected?
print("\n=== Trips dengan LocationID 57 atau 105 ===")
affected = df_clean[
    df_clean["PULocationID"].isin([57, 105]) |
    df_clean["DOLocationID"].isin([57, 105])
]
print(f"Affected trips: {len(affected):,}")
print(affected[["PULocationID", "DOLocationID"]].value_counts().head(10))

Lookup before dedup: 263
Lookup after dedup : 260

=== LocationID 57 dan 105 di GeoJSON ===
Empty DataFrame
Columns: [locationid, zone, borough]
Index: []

=== Trips dengan LocationID 57 atau 105 ===
Affected trips: 36
PULocationID  DOLocationID
138           57              11
132           57               6
263           57               2
82            57               2
93            57               2
7             57               1
79            57               1
57            93               1
186           57               1
231           57               1
Name: count, dtype: int64


In [12]:
# Cell 10 — Re-run join dengan lookup_deduped + drop null H3

# Join pickup
df_enriched = df_clean.merge(
    lookup_deduped[["LocationID", "h3_res8"]].rename(columns={"h3_res8": "h3_pickup_res8"}),
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
).drop(columns=["LocationID"])

# Join dropoff
df_enriched = df_enriched.merge(
    lookup_deduped[["LocationID", "h3_res8"]].rename(columns={"h3_res8": "h3_dropoff_res8"}),
    left_on="DOLocationID",
    right_on="LocationID",
    how="left"
).drop(columns=["LocationID"])

# Derive columns
df_enriched["day_of_week"] = df_enriched["tpep_pickup_datetime"].dt.dayofweek
df_enriched["hour_of_day"] = df_enriched["tpep_pickup_datetime"].dt.hour

# Drop Hive partition columns
df_enriched = df_enriched.drop(columns=["year", "month"])

# Drop 36 rows dengan null H3 (LocationID 57/105 — tidak ada di GeoJSON)
df_enriched = df_enriched.dropna(subset=["h3_pickup_res8", "h3_dropoff_res8"])

print(f"Final rows    : {len(df_enriched):,}")
print(f"Null h3_pickup : {df_enriched['h3_pickup_res8'].isna().sum()}")
print(f"Null h3_dropoff: {df_enriched['h3_dropoff_res8'].isna().sum()}")
print(f"\nSample output:")
print(df_enriched[["PULocationID", "h3_pickup_res8", "DOLocationID", "h3_dropoff_res8", "day_of_week", "hour_of_day"]].head(5))

Final rows    : 2,823,739
Null h3_pickup : 0
Null h3_dropoff: 0

Sample output:
   PULocationID   h3_pickup_res8  DOLocationID  h3_dropoff_res8  day_of_week  \
0           161  882a100d67fffff           141  882a100d69fffff            6   
1            43  882a100895fffff           237  882a100d69fffff            6   
2            48  882a100d65fffff           238  882a100883fffff            6   
3           107  882a100d23fffff            79  882a100d35fffff            6   
4           161  882a100d67fffff           137  882a100d2bfffff            6   

   hour_of_day  
0            0  
1            0  
2            0  
3            0  
4            0  


In [13]:
# Cell 11 — Final validation sebelum prototype dinyatakan done

# 1. Schema check
print("=== Schema ===")
print(df_enriched.dtypes)

# 2. H3 distribution — top 10 busiest pickup cells
print("\n=== Top 10 H3 Pickup Cells ===")
print(df_enriched["h3_pickup_res8"].value_counts().head(10))

# 3. Day of week distribution
print("\n=== Trips by Day of Week (0=Mon, 6=Sun) ===")
print(df_enriched["day_of_week"].value_counts().sort_index())

# 4. Hour of day distribution
print("\n=== Trips by Hour of Day ===")
print(df_enriched["hour_of_day"].value_counts().sort_index())

# 5. Supply-demand ratio preview (D-037 proxy)
print("\n=== Supply-Demand Preview (sample 5 H3 cells) ===")
sample_cells = df_enriched["h3_pickup_res8"].value_counts().head(5).index.tolist()
agg = df_enriched[df_enriched["h3_pickup_res8"].isin(sample_cells)].groupby(
    ["h3_pickup_res8", "day_of_week", "hour_of_day"]
).agg(
    pickups=("PULocationID", "count"),
    dropoffs=("DOLocationID", "count")
).reset_index()
agg["supply_demand_ratio"] = agg["dropoffs"] / agg["pickups"]
print(agg.head(10))

=== Schema ===
VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                     float64
h3_pickup_res8                      str
h3_dropoff_res8                     str
day_of_week                       int32
hour_of_day                       int32
dtype: object

=== Top 10 H3 Pickup Cells ===
h3_pickup_res8
882a

    h3_pickup_res8  day_of_week  hour_of_day  pickups  dropoffs  \
0  882a100d2dfffff            0            0      449       449   
1  882a100d2dfffff            0            1      265       265   
2  882a100d2dfffff            0            2      216       216   
3  882a100d2dfffff            0            3      136       136   
4  882a100d2dfffff            0            4      182       182   
5  882a100d2dfffff            0            5      375       375   
6  882a100d2dfffff            0            6     1140      1140   
7  882a100d2dfffff            0            7     1225      1225   
8  882a100d2dfffff            0            8     1417      1417   
9  882a100d2dfffff            0            9     1580      1580   

   supply_demand_ratio  
0                  1.0  
1                  1.0  
2                  1.0  
3                  1.0  
4                  1.0  
5                  1.0  
6                  1.0  
7                  1.0  
8                  1.0  
9           

In [14]:
# Cell 12a — Day of week + Hour of day
print("=== Trips by Day of Week (0=Mon, 6=Sun) ===")
print(df_enriched["day_of_week"].value_counts().sort_index())

print("\n=== Trips by Hour of Day ===")
print(df_enriched["hour_of_day"].value_counts().sort_index())

=== Trips by Day of Week (0=Mon, 6=Sun) ===
day_of_week
0    371587
1    452999
2    384409
3    406736
4    400882
5    408799
6    398327
Name: count, dtype: int64

=== Trips by Hour of Day ===
hour_of_day
0      77713
1      54398
2      37943
3      24432
4      15416
5      15684
6      38966
7      78001
8     106000
9     120108
10    132358
11    142287
12    157169
13    165226
14    176903
15    181619
16    180862
17    193367
18    199910
19    178874
20    153911
21    149950
22    136837
23    105805
Name: count, dtype: int64


In [15]:
# Cell 12b — Top 10 H3 cells
print("=== Top 10 H3 Pickup Cells ===")
print(df_enriched["h3_pickup_res8"].value_counts().head(10))

=== Top 10 H3 Pickup Cells ===
h3_pickup_res8
882a100d69fffff    216774
882a100d2dfffff    213974
882a100d65fffff    172010
882a103b03fffff    145756
882a100d63fffff    135897
882a100891fffff    130370
882a100d67fffff    128413
882a107253fffff    114194
882a10089bfffff    101899
882a1008bbfffff     94553
Name: count, dtype: int64


In [16]:
# Cell 12c — Supply demand preview
sample_cells = df_enriched["h3_pickup_res8"].value_counts().head(5).index.tolist()
agg = df_enriched[df_enriched["h3_pickup_res8"].isin(sample_cells)].groupby(
    ["h3_pickup_res8", "day_of_week", "hour_of_day"]
).agg(
    pickups=("PULocationID", "count"),
    dropoffs=("DOLocationID", "count")
).reset_index()
agg["supply_demand_ratio"] = agg["dropoffs"] / agg["pickups"]
print(agg.head(15))

     h3_pickup_res8  day_of_week  hour_of_day  pickups  dropoffs  \
0   882a100d2dfffff            0            0      449       449   
1   882a100d2dfffff            0            1      265       265   
2   882a100d2dfffff            0            2      216       216   
3   882a100d2dfffff            0            3      136       136   
4   882a100d2dfffff            0            4      182       182   
5   882a100d2dfffff            0            5      375       375   
6   882a100d2dfffff            0            6     1140      1140   
7   882a100d2dfffff            0            7     1225      1225   
8   882a100d2dfffff            0            8     1417      1417   
9   882a100d2dfffff            0            9     1580      1580   
10  882a100d2dfffff            0           10     1576      1576   
11  882a100d2dfffff            0           11     1616      1616   
12  882a100d2dfffff            0           12     1680      1680   
13  882a100d2dfffff            0           13   

In [17]:
# Cell 13 — Correct supply-demand aggregation (D-037)
sample_cells = df_enriched["h3_pickup_res8"].value_counts().head(5).index.tolist()

# Demand: pickup events per H3 pickup cell per time bucket
demand = df_enriched.groupby(
    ["h3_pickup_res8", "day_of_week", "hour_of_day"]
).agg(
    pickups=("PULocationID", "count")
).reset_index()

# Supply proxy: dropoff events per H3 dropoff cell per time bucket
supply = df_enriched.groupby(
    ["h3_dropoff_res8", "day_of_week", "hour_of_day"]
).agg(
    dropoffs=("DOLocationID", "count")
).reset_index().rename(columns={"h3_dropoff_res8": "h3_cell"})

demand = demand.rename(columns={"h3_pickup_res8": "h3_cell"})

# Join supply + demand on same H3 cell + time bucket
gap = demand.merge(supply, on=["h3_cell", "day_of_week", "hour_of_day"], how="outer").fillna(0)
gap["supply_demand_ratio"] = gap["dropoffs"] / gap["pickups"].replace(0, float("nan"))

# Preview sample cells
print(gap[gap["h3_cell"].isin(sample_cells)].sort_values(
    ["h3_cell", "day_of_week", "hour_of_day"]
).head(15).to_string())

               h3_cell  day_of_week  hour_of_day  pickups  dropoffs  supply_demand_ratio
14512  882a100d2dfffff            0            0    449.0     261.0             0.581292
14513  882a100d2dfffff            0            1    265.0     129.0             0.486792
14514  882a100d2dfffff            0            2    216.0      75.0             0.347222
14515  882a100d2dfffff            0            3    136.0      55.0             0.404412
14516  882a100d2dfffff            0            4    182.0      54.0             0.296703
14517  882a100d2dfffff            0            5    375.0      88.0             0.234667
14518  882a100d2dfffff            0            6   1140.0     323.0             0.283333
14519  882a100d2dfffff            0            7   1225.0     556.0             0.453878
14520  882a100d2dfffff            0            8   1417.0     889.0             0.627382
14521  882a100d2dfffff            0            9   1580.0    1106.0             0.700000
14522  882a100d2dffff